# AI012 – M2 Training Notebook: Local Outlier Factor

**Role E | Preetham Chandu**

This notebook trains the M2 (LOF) anomaly detection model using:
- **Role A dataset**: `anomaly_detection_hourly_2020_2024.csv` (Sunain)
- **Role B features**: `ai004_features_output.csv` (Sneha) — merged if available, computed from raw if not
- **Role C labels**: `anomaly_labels_v1.csv` (Preetham) — used for validation only, NOT training

All reusable logic lives in `src/models/m2.py`. This notebook is for training and parameter tuning only.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'src'))
from models.m2 import LOFAnomalyModel, load_data, build_output, save_outputs

## 1. Load Data (Role A + Role B)

In [2]:
repo_root    = Path.cwd().parents[3]
dataset_path = repo_root / 'ai-ml' / 'datasets' / 'anomaly_detection_hourly_2020_2024.csv'

# Role B external feature CSV intentionally disabled.
# M2 computes engineered features internally from raw columns.
df = load_data(str(dataset_path), None)

print(f'[M2-LOF] Final shape: {df.shape}')
df.head()


[M2-LOF] Final shape: (169539, 52)


,time_window,region_id,region_lat_bin,region_lon_bin,region_min_latitude,region_max_latitude,region_min_longitude,region_max_longitude,firms_event_count,firms_avg_latitude,...,urlhaus_links,urlhaus_reporters,urlhaus_hour_sources,cyber_intensity,hazard_severity,rolling_cyber_mean,cyber_zscore,hazard_zscore,combined_risk_index,temporal_spike
0,2020-01-02 03:00:00,lat_-2_lon_28,-2,28,-10.0,-5.0,140.0,145.0,2,-9.396850,...,NaN,NaN,NaN,0.0,0.009726,0.0,-0.062369,-0.108562,0.004863,0.0
1,2020-10-05 03:00:00,lat_-2_lon_28,-2,28,-10.0,-5.0,140.0,145.0,1,-9.396970,...,NaN,NaN,NaN,0.0,0.008282,0.0,-0.062369,-0.203836,0.004141,0.0
2,2020-10-05 04:00:00,lat_-2_lon_28,-2,28,-10.0,-5.0,140.0,145.0,2,-9.401175,...,NaN,NaN,NaN,0.0,0.008851,0.0,-0.062369,-0.166271,0.004426,0.0
3,2021-12-31 03:00:00,lat_-2_lon_28,-2,28,-10.0,-5.0,140.0,145.0,2,-9.396310,...,NaN,NaN,NaN,0.0,0.005494,0.0,-0.062369,-0.387852,0.002747,0.0
4,2022-01-01 04:00:00,lat_-2_lon_28,-2,28,-10.0,-5.0,140.0,145.0,2,-9.383980,...,NaN,NaN,NaN,0.0,0.003217,0.0,-0.062369,-0.538113,0.001609,0.0


## 2. Parameter Tuning — n_neighbors

Test n_neighbors = 5, 10, 15, 20 to find the cleanest score range.

In [3]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

FEATURES = [
    'firms_event_count','firms_avg_frp','firms_avg_confidence',
    'urlhaus_event_count','urlhaus_online_count',
    'region_lat_bin','region_lon_bin',
    'cyber_intensity','hazard_severity','cyber_zscore','hazard_zscore',
    'combined_risk_index','temporal_spike'
]
feat_cols = [c for c in FEATURES if c in df.columns]
X = df[feat_cols].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Testing n_neighbors...')
for n in [5, 10, 15, 20]:
    lof = LocalOutlierFactor(n_neighbors=n, contamination=0.07, novelty=False, n_jobs=-1)
    preds = lof.fit_predict(X_scaled)
    scores = -lof.negative_outlier_factor_
    flagged = (preds==-1).sum()
    note = '← SELECTED: clean, interpretable scores' if n==20 else '← REJECTED: unrealistic scores from duplicate distances' if n==5 else '← REJECTED: still artefacts'
    print(f'n={n:<2}: flagged={flagged} ({flagged/len(df)*100:.2f}%)  score_max={scores.max():>14.2f}  {note}')

Testing n_neighbors...
n=5 : flagged=11868 (7.00%)  score_max=   42578953.33  ← REJECTED: unrealistic scores from duplicate distances
n=10: flagged=11868 (7.00%)  score_max=   14124857.85  ← REJECTED: still artefacts
n=15: flagged=11868 (7.00%)  score_max=   14797470.08  ← REJECTED: still artefacts
n=20: flagged=11868 (7.00%)  score_max=         42.21  ← SELECTED: clean, interpretable scores


## 3. Parameter Tuning — contamination

Test contamination values. Role C labels give us ~6.74% as the expected rate.

In [4]:
print('Testing contamination with n_neighbors=20...')
for c in [0.01, 0.03, 0.05, 0.07, 0.10]:
    lof = LocalOutlierFactor(n_neighbors=20, contamination=c, novelty=False, n_jobs=-1)
    preds = lof.fit_predict(X_scaled)
    flagged = (preds==-1).sum()
    note = '← SELECTED: aligns with Role C label rate 6.74%' if c==0.07 else ''
    print(f'contamination={c}: flagged={flagged:>6} ({flagged/len(df)*100:.2f}%)  {note}')

Testing contamination with n_neighbors=20...
contamination=0.01: flagged=  1696 (1.00%)  
contamination=0.03: flagged=  5087 (3.00%)  
contamination=0.05: flagged=  8477 (5.00%)  
contamination=0.07: flagged= 11868 (7.00%)  ← SELECTED: aligns with Role C label rate 6.74%
contamination=0.1: flagged= 16954 (10.00%)  


## 4. Train Final Model (n_neighbors=20, contamination=0.07)

In [5]:
model = LOFAnomalyModel(n_neighbors=20, contamination=0.07)
lof_flags, lof_scores = model.train(df)

## 5. Severity Breakdown

In [6]:
predictions = build_output(df, lof_flags, lof_scores)
print('Severity breakdown:')
print(predictions['lof_severity'].value_counts().sort_index().to_string())
print('Anomaly (non-normal):', (predictions['lof_flag']==1).sum())

Severity breakdown:
lof_severity
critical       138
high           416
low           3209
medium        1042
normal      164734
Anomaly (non-normal): 10514


## 6. Save Checkpoint (m2.pkl) and Predictions

In [7]:
model.save_checkpoint('../checkpoints/m2.pkl')
save_outputs(predictions, '../data/processed/lof_predictions.csv')